In [29]:
import json
import os
import sys
sys.path
sys.path.append('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer')
import pandas as pd

In [30]:
def generate_examples( statements_file, all_csv_path):
    def convert_to_table_structure(table_str):
        header = table_str.split("\n")[0].split("#")
        rows = [row.split("#") for row in table_str.strip().split("\n")[1:]]
        return {"header": header, "rows": rows}

    with open(statements_file, encoding="utf-8") as f:
        examples = json.load(f)

    for i, (table_id, example) in enumerate(examples.items()):
        table_file_path = os.path.join(all_csv_path, table_id)
        with open(table_file_path, encoding="utf-8") as f:
            table_text = f.read()

        statements, labels, caption = example

        for statement_idx, (statement, label) in enumerate(zip(statements, labels)):
            table = convert_to_table_structure(table_text)
            #yield (
             #   f"{i}_{statement_idx}",
              #  {
               #     "id": i,
                #    "table": {
                 #       "id": table_id,
                  #      "header": table["header"],
                   #     "rows": table["rows"],
                    #    "caption": caption,
                    #},
          #          "statement": statement,
           #         "label": label,
            #    },
            #)
            yield {
                "id":i,
                "table_csv":table_id,
                "table_text":table_text,
                "label":label,
                "statement":statement,
                "table_caption": caption
                
            }


In [3]:
tabfact = TabFact()

In [3]:
tabfact

In [32]:
all_csv_path = '../../Table-Fact-Checking/data/all_csv'
test_map = '../../Table-Fact-Checking/tokenized_data/test_examples.json'
train_map = '../../Table-Fact-Checking/tokenized_data/train_examples.json'
val_map = '../../Table-Fact-Checking/tokenized_data/val_examples.json'

In [8]:
tabfact._generate_examples(test_map,all_csv_path)


<generator object TabFact._generate_examples at 0x7a8ec885fd80>

In [33]:
from datasets import Dataset

In [24]:
ds = Dataset.from_list(list(generate_examples(test_map,all_csv_path)))

In [25]:
ds.save_to_disk('tab_fact_test')

Saving the dataset (1/1 shards): 100%|██████████| 12779/12779 [00:00<00:00, 382631.43 examples/s]


In [34]:
ds = Dataset.from_list(list(generate_examples(train_map,all_csv_path)))
ds.save_to_disk('tab_fact_train')

Saving the dataset (1/1 shards): 100%|██████████| 92283/92283 [00:00<00:00, 1037513.15 examples/s]


In [35]:
ds = Dataset.from_list(list(generate_examples(val_map,all_csv_path)))
ds.save_to_disk('tab_fact_val')

Saving the dataset (1/1 shards): 100%|██████████| 12792/12792 [00:00<00:00, 824906.01 examples/s]


In [27]:
from datasets import load_from_disk
data = load_from_disk('tab_fact_test_xml/')

In [28]:
data

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query'],
    num_rows: 12779
})

In [ ]:
python zeroshot_deepseek_test.py --query-field semtab --input_data ./tab_fact_test_xml --output_data ./tab_fact_test_xml_semtab --num-proc 16 > gen_answ_sem_tab_fact_test.log

In [ ]:
python zeroshot_deepseek_test.py --query-field nlsep --input_data ./tab_fact_test_xml --output_data ./tab_fact_test_xml_nlsep --num-proc 16 > gen_answ_nlsep_tab_fact_test.log